In [2]:
from non_rigid.nets.pn2 import PN2Dense
from non_rigid.nets.dgcnn import DGCNN
from non_rigid.datasets.dedo import DedoDataset, DedoDataModule
import numpy as np
import torch
from omegaconf import OmegaConf
import json
import os
from pathlib import Path

import torch_geometric.data as tgd
import torch_geometric.loader as tgl

import rpad.visualize_3d.plots as vpl
from plotly import graph_objects as go
from plotly.subplots import make_subplots

from tqdm import tqdm

# ignore TypedStorage warnings
import warnings
warnings.filterwarnings("ignore", message="TypedStorage is deprecated", category=UserWarning)

In [3]:
# torch settings
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Since most of us are training on 3090s+, we can use mixed precision.
torch.set_float32_matmul_precision("medium")
torch.manual_seed(42)

In [4]:
# set dataset config and create datamodule
dataset_cfg = OmegaConf.load("../configs/dataset/dedo.yaml")

overrides = {
    "train_size": 10, #400,
    "scene": False,
    "world_frame": False,
    "scene_anchor": False,
    "rel_pose": False,
    "rel_pose_type": "translation",
    "center_type": "anchor_center",
    "predict_ref_frame": True,
    "action_context_center_type": "center",
    "cloth_geometry": "multi",
    "cloth_pose": "random",
    # "scene_transform_type": "random_flat_upright",
    "sample_size_anchor": 1024,
}

dataset_cfg = OmegaConf.merge(dataset_cfg, overrides)

# print(
#     json.dumps(
#         OmegaConf.to_container(dataset_cfg, resolve=True, throw_on_missing=False),
#         sort_keys=True,
#         indent=4,
#     )
# )

datamodule = DedoDataModule(
    batch_size=2, # 16,
    val_batch_size=4,
    num_workers=1,
    dataset_cfg=dataset_cfg,
)
datamodule.setup("fit")



class PygDataset(tgd.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
    
    def __len__(self):
        return len(self.dataset)
    
    def get(self, index):
        """
        Mini-wrapper for DedoDataset to return torch_geometric.data.Data. 
        "pos" is just the anchor point cloud, and "y" is the mean of the goal action point cloud in the anchor frame.
        """
        item = self.dataset[index]
        num_anchor_points = item["pc_anchor"].shape[0]

        # adding small random noise to anchor point cloud
        # item["pc_anchor"] += 5e-1 * torch.randn_like(item["pc_anchor"])

        data = tgd.Data(
            pos=item["pc_anchor"],
            y=item["pc"].mean(dim=0, keepdim=True).repeat(num_anchor_points, 1),
        )
        return data
    
    def __getitem__(self, index):
        return self.get(index)


train_dataset = PygDataset(datamodule.train_dataset)
val_dataset = PygDataset(datamodule.val_dataset)

train_batch_size = 16
val_batch_size = 4

train_loader = tgl.DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, num_workers=16)
val_loader = tgl.DataLoader(val_dataset, batch_size=val_batch_size, shuffle=False, num_workers=16)

In [5]:
d1 = train_dataset[0]
d2 = train_dataset[1]
d3 = val_dataset[0]
d4 = val_dataset[1]

d1points = torch.cat([d1.pos, d1.y], dim=0)
d2points = torch.cat([d2.pos, d2.y], dim=0)
d3points = torch.cat([d3.pos, d3.y], dim=0)
d4points = torch.cat([d4.pos, d4.y], dim=0)

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=d1points[:, 0],
        y=d1points[:, 1],
        z=d1points[:, 2],
        mode="markers",
        marker=dict(size=5),
    )
)
fig.add_trace(
    go.Scatter3d(
        x=d2points[:, 0],
        y=d2points[:, 1],
        z=d2points[:, 2],
        mode="markers",
        marker=dict(size=5),
    )
)
fig.add_trace(
    go.Scatter3d(
        x=d3points[:, 0],
        y=d3points[:, 1],
        z=d3points[:, 2],
        mode="markers",
        marker=dict(size=5),
    )
)
fig.add_trace(
    go.Scatter3d(
        x=d4points[:, 0],
        y=d4points[:, 1],
        z=d4points[:, 2],
        mode="markers",
        marker=dict(size=5),
    )
)
fig.show(renderer="browser")

In [6]:
# creating model

class FramePredictorSimple(torch.nn.Module):
    def __init__(self, out_channels):
        super(FramePredictorSimple, self).__init__()
        self.pn2 = PN2Dense(in_channels=0, out_channels=out_channels)

    def forward(self, batch):
        # make sure all of the point clouds in the batch have the same number of points
        ptr_diffs = torch.unique(batch.ptr[1:] - batch.ptr[:-1])
        if len(ptr_diffs) > 1:
            raise ValueError("All point clouds in the batch must have the same number of points.")
        else:
            num_points = ptr_diffs.item()

        output = self.pn2(batch)
        logits = output[:, [0]]
        residuals = output[:, 1:4]
        vars = output[:, 4:]

        # run vars through softplus to ensure positive values
        vars = torch.nn.functional.softplus(vars)

        # add mean residuals to points to get mean predictions
        means = residuals + batch.pos

        # converting logits to probabilities
        probs = torch.nn.functional.softmax(logits.reshape(-1, num_points), dim=1).reshape(-1, 1)

        return {
            "probs": probs,
            "means": means,
            "vars": vars,
        }
    

class FramePredictorDGCNN(torch.nn.Module):
    def __init__(self, out_channels):
        super(FramePredictorDGCNN, self).__init__()
        self.out_channels = out_channels
        self.dgcnn = DGCNN(emb_dims=512)
        self.final = torch.nn.Conv1d(
            in_channels=512,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True,
        )

    def forward(self, batch):
        # make sure all of the point clouds in the batch have the same number of points
        ptr_diffs = torch.unique(batch.ptr[1:] - batch.ptr[:-1])
        if len(ptr_diffs) > 1:
            raise ValueError("All point clouds in the batch must have the same number of points.")
        else:
            num_points = ptr_diffs.item()
        
        input = batch.pos.reshape(-1, 3, num_points)
        output = self.dgcnn(input)
        output = self.final(output)

        # reshape output to get logits, residuals, and vars consistent with tgd.Batch
        output = output.permute(0, 2, 1).reshape(-1, self.out_channels)

        logits = output[:, [0]]
        residuals = output[:, 1:4]
        vars = output[:, 4:]

        # run vars through softplus to ensure positive values
        vars = torch.nn.functional.softplus(vars)

        # add mean residuals to points to get mean predictions
        means = residuals
        # means = residuals + batch.pos

        # converting logits to probabilities
        probs = torch.nn.functional.softmax(logits.reshape(-1, num_points), dim=1).reshape(-1, 1)

        return {
            "probs": probs,
            "means": means,
            "vars": vars,
        }


In [7]:
# defining GMM loss function
class GMMLoss(torch.nn.Module):
    def __init__(self, eps=1e-6):
        """
        eps: value used to clamp var, for stability.
        """
        super(GMMLoss, self).__init__()
        self.eps = eps
    
    def forward(self, batch, pred):
        """
        batch: torch_geometric.data.Batch object. batch.y is (N, 3) tensor of target means.
        pred: dict with keys "probs", "means", "vars".
        """
        # make sure all of the point clouds in the batch have the same number of points
        ptr_diffs = torch.unique(batch.ptr[1:] - batch.ptr[:-1])
        if len(ptr_diffs) > 1:
            raise ValueError("All point clouds in the batch must have the same number of points.")
        else:
            num_points = ptr_diffs.item()

        targets = batch.y
        probs = pred["probs"]
        means = pred["means"]
        vars = pred["vars"]


        # reshape into (N, num_points, 3)
        targets = targets.reshape(-1, num_points, 3)
        means = means.reshape(-1, num_points, 3)
        vars = vars.reshape(-1, num_points, 1)
        probs = probs.reshape(-1, num_points, 1)


        # clamp vars for stability
        # vars = torch.clamp(vars, min=self.eps)
        # probs = 1.0 / 1024.0
        probs = 1.0 / num_points
        vars = 0.01

        # # multivariate homoscedastic gaussian likelihood
        # # norm_const = 1 / torch.sqrt(2 * np.pi**3 * vars)
        # norm_const = 1 / np.sqrt(2 * np.pi)**3
        # diff = targets - means
        # likelihood = torch.exp(-0.5 * torch.sum(diff ** 2 / vars, dim=1, keepdim=True))
        # likelihood = likelihood * norm_const



        # manual logsumexp for stability, accounting for normalization constant and per-point weights
        # need to account for variance prediction eventually
        point_likelihood_exps = -0.5 * torch.sum((targets - means) ** 2 / vars, dim=-1, keepdim=True)
        maxlog = point_likelihood_exps.max(dim=-2, keepdim=True).values
        point_likelihoods = torch.exp(point_likelihood_exps - maxlog)
        likelihoods = torch.sum(point_likelihoods * probs, dim=-2, keepdim=True)
        log_likelihoods = torch.log(likelihoods) + maxlog
        
        loss = -torch.mean(log_likelihoods)
        
        # # weighted sum of likelihoods across point cloud
        # likelihood = likelihood * probs

        # # log of sum of likelihoods per point cloud
        # likelihood = likelihood.reshape(-1, num_points).sum(dim=1)
        # loss = -torch.log(likelihood).mean()
        return loss

In [17]:
batch = tgd.Batch.from_data_list([d1])
batch.y.shape
batch.y

means = torch.tensor([[-2.0553, -2.8745, 7.8815]]*1024).to(batch.pos.device)
pred = {
    "probs": torch.zeros(1024, 1).to(batch.pos.device),
    "means": means,
    "vars": torch.zeros(2048, 1).to(batch.pos.device),
}

batch.y


means.shape

torch.Size([1024, 3])

In [7]:
# create model - currently 5 for heteroscedastic variance
# model = FramePredictorSimple(5)
model = FramePredictorDGCNN(5)

# set up optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)

# initialize loss
loss_fn = GMMLoss()

# training params
num_epochs = 5000
val_every = 50

device = "cuda:0"
model.to(device)

total_losses = []
total_val_losses = []

min_val_loss = float("inf")





# visualize some predictions if val loss is low
val_fig = make_subplots(rows=2, cols=3, 
                        specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "scatter3d"}], 
                                [{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "scatter3d"}]],
)

# visualize train
for i in range(3):
    vis_batch = tgd.Batch.from_data_list([train_dataset[i]]).to(device)
    with torch.no_grad():
        pred = model(vis_batch)
    probs = pred["probs"].cpu().numpy()
    target = vis_batch.y.cpu().numpy()[0]
    pc_anchor = vis_batch.pos.cpu().numpy()
    means = pred["means"].cpu().numpy()
    val_fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker=dict(size=2, color="blue"),
            x=pc_anchor[:, 0],
            y=pc_anchor[:, 1],
            z=pc_anchor[:, 2],
        ), row=1, col=i + 1
    )
    val_fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker=dict(size=2, color=probs, colorscale="Inferno", colorbar=dict(title="Prob")),
            x=means[:, 0],
            y=means[:, 1],
            z=means[:, 2],
        ), row=1, col=i + 1
    )
    val_fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker=dict(size=5, color="green"),
            x=[target[0]],
            y=[target[1]],
            z=[target[2]],
        ), row=1, col=i + 1
    )
    # val_fig.add_traces(viz_traces(vis_batch, pred), rows=1, cols=i + 1)

# visualize val
for i in range(3):
    vis_batch = tgd.Batch.from_data_list([val_dataset[i]]).to(device)
    with torch.no_grad():
        pred = model(vis_batch)
    probs = pred["probs"].cpu().numpy()
    target = vis_batch.y.cpu().numpy()[0]
    pc_anchor = vis_batch.pos.cpu().numpy()
    means = pred["means"].cpu().numpy()
    val_fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker=dict(size=2, color="blue"),
            x=pc_anchor[:, 0],
            y=pc_anchor[:, 1],
            z=pc_anchor[:, 2],
        ), row=2, col=i + 1
    )
    val_fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker=dict(size=2, color=probs, colorscale="Inferno", colorbar=dict(title="Prob")),
            x=means[:, 0],
            y=means[:, 1],
            z=means[:, 2],
            showlegend=True,
        ), row=2, col=i + 1
    )
    val_fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker=dict(size=5, color="green"),
            x=[target[0]],
            y=[target[1]],
            z=[target[2]],
        ), row=2, col=i + 1
    )
    # val_fig.add_traces(viz_traces(vis_batch, pred), rows=2, cols=i + 1)

# visualize predictions
val_fig.update_layout(title_text="Epoch 0, Train Loss: N/A Val Loss: N/A")
val_fig.show(renderer="browser")





# basic training loop
for epoch in range(num_epochs):
    # train step
    model.train()
    epoch_loss = []
    for i, batch in enumerate(train_loader):
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        #print(i, pred.shape, batch.ptr)

        loss = loss_fn(batch, pred)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss.append(loss.item())
    epoch_loss = np.mean(epoch_loss)
    total_losses.append(epoch_loss)

    # val step
    if (epoch + 1) % val_every == 0:
        model.eval()
        val_loss = []
        with torch.no_grad():
            for i, batch in enumerate(val_loader):
                batch = batch.to(device)
                pred = model(batch)

                loss = loss_fn(batch, pred)
                val_loss.append(loss.item())
        val_loss = np.mean(val_loss)
        total_val_losses.append(val_loss)

    # logging.permute(1, 0).reshape(6, -1).permute(1, 0)
    if (epoch + 1) % val_every == 0:
        print(f"Epoch {epoch}, Train Loss: {epoch_loss}, Val Loss: {val_loss}")

        # visualize some predictions if val loss is low
        val_fig = make_subplots(rows=2, cols=3, 
                                specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "scatter3d"}], 
                                       [{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "scatter3d"}]],
        )

        # visualize train
        for i in range(3):
            vis_batch = tgd.Batch.from_data_list([train_dataset[i]]).to(device)
            with torch.no_grad():
                pred = model(vis_batch)
            probs = pred["probs"].cpu().numpy()
            target = vis_batch.y.cpu().numpy()[0]
            pc_anchor = vis_batch.pos.cpu().numpy()
            means = pred["means"].cpu().numpy()
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=2, color="blue"),
                    x=pc_anchor[:, 0],
                    y=pc_anchor[:, 1],
                    z=pc_anchor[:, 2],
                ), row=1, col=i + 1
            )
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=2, color=probs, colorscale="Inferno", colorbar=dict(title="Prob")),
                    x=means[:, 0],
                    y=means[:, 1],
                    z=means[:, 2],
                ), row=1, col=i + 1
            )
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=5, color="green"),
                    x=[target[0]],
                    y=[target[1]],
                    z=[target[2]],
                ), row=1, col=i + 1
            )
            # val_fig.add_traces(viz_traces(vis_batch, pred), rows=1, cols=i + 1)
        
        # visualize val
        for i in range(3):
            vis_batch = tgd.Batch.from_data_list([val_dataset[i]]).to(device)
            with torch.no_grad():
                pred = model(vis_batch)
            probs = pred["probs"].cpu().numpy()
            target = vis_batch.y.cpu().numpy()[0]
            pc_anchor = vis_batch.pos.cpu().numpy()
            means = pred["means"].cpu().numpy()
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=2, color="blue"),
                    x=pc_anchor[:, 0],
                    y=pc_anchor[:, 1],
                    z=pc_anchor[:, 2],
                ), row=2, col=i + 1
            )
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=2, color=probs, colorscale="Inferno", colorbar=dict(title="Prob")),
                    x=means[:, 0],
                    y=means[:, 1],
                    z=means[:, 2],
                    showlegend=True,
                ), row=2, col=i + 1
            )
            val_fig.add_trace(
                go.Scatter3d(
                    mode="markers",
                    marker=dict(size=5, color="green"),
                    x=[target[0]],
                    y=[target[1]],
                    z=[target[2]],
                ), row=2, col=i + 1
            )
            # val_fig.add_traces(viz_traces(vis_batch, pred), rows=2, cols=i + 1)

        # visualize predictions
        val_fig.update_layout(title_text=f"Epoch {epoch}, Train Loss: {epoch_loss} Val Loss: {val_loss}")
        val_fig.show(renderer="browser")
    else:
        print(f"Epoch {epoch}, Train Loss: {epoch_loss}")


# After training, plot the losses
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(1, len(total_losses) + 1), y=total_losses, name="Train Loss"))
fig.add_trace(go.Scatter(x=np.arange(val_every, (len(total_val_losses) + 1) * val_every, val_every), y=total_val_losses, name="Val Loss"))
fig.update_layout(title="Losses", xaxis_title="Epoch", yaxis_title="Loss")
fig.show()


Epoch 0, Train Loss: 3610.572265625
Epoch 1, Train Loss: 131.3885498046875
Epoch 2, Train Loss: 210.6193389892578
Epoch 3, Train Loss: 241.5878448486328
Epoch 4, Train Loss: 17.069799423217773
Epoch 5, Train Loss: 142.99130249023438
Epoch 6, Train Loss: 42.78425216674805
Epoch 7, Train Loss: 40.95233154296875
Epoch 8, Train Loss: 12.87384033203125
Epoch 9, Train Loss: 38.112037658691406
Epoch 10, Train Loss: 63.88798904418945
Epoch 11, Train Loss: 44.008174896240234
Epoch 12, Train Loss: 36.674869537353516
Epoch 13, Train Loss: 84.78189086914062
Epoch 14, Train Loss: 64.871337890625
Epoch 15, Train Loss: 146.89830017089844
Epoch 16, Train Loss: 33.859718322753906
Epoch 17, Train Loss: 10.76414966583252
Epoch 18, Train Loss: 22.162254333496094
Epoch 19, Train Loss: 15.384509086608887
Epoch 20, Train Loss: 8.158665657043457
Epoch 21, Train Loss: 9.44931411743164
Epoch 22, Train Loss: 13.473388671875
Epoch 23, Train Loss: 25.687789916992188
Epoch 24, Train Loss: 26.037220001220703
Epoch 2

In [ ]:
# visualization code
model.eval()


for i in range(4):
    batch = tgd.Batch.from_data_list([val_dataset[i]]).to(device)


    with torch.no_grad():
        output = model(batch)

    # logits = output["logits"].cpu().numpy()
    means = output["means"].cpu().numpy()
    vars = output["vars"].cpu().numpy()
    # vars = np.linalg.norm(vars, axis=1)

    pc_anchor = batch.pos.cpu().numpy()

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker={
                "size": 5,
                "color": "blue",
            },
            x=pc_anchor[:, 0],
            y=pc_anchor[:, 1],
            z=pc_anchor[:, 2],
        )
    )
    fig.add_trace(
        go.Scatter3d(
            mode="markers",
            marker={
                "size": 5,
                "color": "green",
            },
            x=means[:, 0],
            y=means[:, 1],
            z=means[:, 2],
        )
    )
    traces = vpl._flow_traces(
        start=pc_anchor,
        flows=means - pc_anchor,
        flowscale=1.0,
        flowcolor="red",
    )
    fig.add_traces(traces[0])
    fig.show(renderer="browser")

In [ ]:
means

array([[-0.0178299 ,  0.02845711,  7.927077  ],
       [-0.0178299 ,  0.02845711,  7.927077  ],
       [-0.0178299 ,  0.02845711,  7.927077  ],
       ...,
       [-0.0178299 ,  0.02845711,  7.927077  ],
       [-0.0178299 ,  0.02845711,  7.927077  ],
       [-0.0178299 ,  0.02845711,  7.927077  ]], dtype=float32)

In [ ]:
# set up model
class FramePredictorSimple(torch.nn.Module):
    def __init__(self, out_channels):
        super().__init__()

        self.pn = PN2Dense(
            in_channels = 0,
            out_channels = out_channels,
        )

    def forward(self, x):
        return self.pn(x)

model = FramePredictorSimple(7)